# Reconstrucción de la Zonificación País 2026

**Propósito:** Reconstruir la hoja `ZonificacionPais` del archivo Metaverso 2026 a partir de:
1. Las hojas de zonificación (`Comunitarios` e `Integrales-Convenios`)
2. El maestro de UDS (`UDS_15052026`)
3. Cupos de tránsito (`TH_Transito_15052026`)
4. Cupos aprobados al 31/12/2025 (`UDS_31122025`)

El resultado final conserva la misma estructura de columnas del original (97 columnas no monetarias)
y agrega 3 columnas nuevas al final: `CuposUDS_31122025`, `CuposServicioUDS_31122025` y `TH_Transito_Cupos`.

In [ ]:
import pandas as pd
import numpy as np
import os
import unicodedata
import warnings
warnings.filterwarnings('ignore')

DIR_BASE = r"D:\ICBF\cost-tracking\data\insumos metaverso"
print(f"Directorio base: {DIR_BASE}")
print(f"Archivos disponibles: {os.listdir(DIR_BASE)}")

---
## 2. Carga y concatenación de hojas de zonificación

Se leen las hojas `Comunitarios` (columnas A–N) e `Integrales-Convenios` (columnas A–J + M)
del archivo de abastecimiento. Ambas se concatenan en un solo DataFrame (`CONCAT`) con un
discriminador `Tipo_Modalidad` (`COMUNITARIO` / `INTEGRAL`).

In [ ]:
INPUT_FILE = os.path.join(DIR_BASE, "zonificación_abastecimiento_servicios_primera_infancia25052026.xlsx")
CONCAT_FILE = os.path.join(DIR_BASE, "CONCAT_ZONIFICACION_METAVERSO_2026.xlsx")

# --- Hoja Comunitarios (A-N) ---
df_com = pd.read_excel(INPUT_FILE, sheet_name="Comunitarios", header=None, skiprows=1)
df_com = df_com.iloc[:, 0:14].copy()
col_map_com = {
    0: "Regional_UDS", 1: "Centro_Zonal_UDS", 2: "Municipio_UDS",
    3: "ZONA", 4: "Codigo_UDS", 5: "UDS",
    6: "LITERAL_DE_CONTRATACION", 7: "LITERAL_DE_CONTRATACION_ALTERNATIVO",
    8: "Servicio", 9: "Atenciones", 10: "Atenciones_30042026",
    11: "Cupos", 12: "Madres_Unds", 13: "Abrev",
}
df_com.rename(columns=col_map_com, inplace=True)
df_com["Tipo_Modalidad"] = "COMUNITARIO"
print(f"Comunitarios: {len(df_com)} filas")

# --- Hoja Integrales-Convenios (A-J + M) ---
df_int = pd.read_excel(INPUT_FILE, sheet_name="Integrales-Convenios", header=None, skiprows=1)
cols_int = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 12]
df_int = df_int.iloc[:, cols_int].copy()
col_map_int = {
    0: "ZONA", 1: "Regional_UDS", 2: "Centro_Zonal_UDS",
    3: "Municipio_UDS", 4: "Codigo_UDS", 5: "UDS",
    6: "LITERAL_DE_CONTRATACION", 7: "LITERAL_DE_CONTRATACION_ALTERNATIVO",
    8: "Servicio", 9: "Componente_para_la_UDS", 12: "Cupos",
}
df_int.rename(columns=col_map_int, inplace=True)
df_int["Tipo_Modalidad"] = "INTEGRAL"
print(f"Integrales-Convenios: {len(df_int)} filas")

# --- Concatenación ---
common_cols = [
    "Regional_UDS", "Centro_Zonal_UDS", "Municipio_UDS", "ZONA",
    "Codigo_UDS", "UDS",
    "LITERAL_DE_CONTRATACION", "LITERAL_DE_CONTRATACION_ALTERNATIVO",
    "Servicio", "Cupos",
]
df_cat = pd.concat([df_com, df_int], axis=0, ignore_index=True, sort=False)
cols_order = common_cols + [
    c for c in df_cat.columns if c not in common_cols and c != "Tipo_Modalidad"
] + ["Tipo_Modalidad"]
df_cat = df_cat[[c for c in cols_order if c in df_cat.columns]]

print(f"\nCONCAT total: {len(df_cat)} filas, {len(df_cat.columns)} columnas")
print(f"Modalidades: {df_cat['Tipo_Modalidad'].value_counts().to_dict()}")
display(df_cat.head(3))
print(f"\nColumnas: {list(df_cat.columns)}")

---
## 3. Enriquecimiento con maestro de UDS

Se hace un `LEFT JOIN` contra la hoja `UDS_15052026` (maestro desduplicado por `CodigoUnidadServicioUDS`).
Las columnas del maestro se integran al DataFrame para completar datos como regional, departamento,
municipio, dirección, estado, vigencia, contratista, etc.

In [ ]:
UDS_FILE = os.path.join(DIR_BASE, "UDS_15052026_3112025.xlsx")

df_cat["k"] = df_cat["Codigo_UDS"].astype(str).str.strip()

uds = pd.read_excel(UDS_FILE, sheet_name="UDS_15052026")
uds["k"] = uds["CodigoUnidadServicioUDS"].astype(str).str.strip()
uds = uds.drop_duplicates(subset="k", keep="first")
print(f"UDS maestro: {len(uds)} registros únicos (de {len(uds) + uds.duplicated(subset='k').sum()} brutos)")

df = df_cat.merge(uds, on="k", how="left", suffixes=("_cat", "_uds"))
print(f"Enlazados: {df['CodigoUnidadServicioUDS'].notna().sum()} / {len(df)}")
print(f"No enlazados: {df['CodigoUnidadServicioUDS'].isna().sum()} / {len(df)}")

---
## 4. Cupos de tránsito (`TH_Transito_15052026`)

Se integra la columna `total` (cupos de tránsito) desde la hoja `TH_Transito_15052026`,
desduplicada por `codigounidadservicio`.

In [ ]:
th = pd.read_excel(UDS_FILE, sheet_name="TH_Transito_15052026")
th["k"] = th["codigounidadservicio"].astype(str).str.strip()
th = th.drop_duplicates(subset="k", keep="first")
print(f"TH_Transito: {len(th)} registros únicos (de {len(th) + th.duplicated(subset='k').sum()} brutos)")

df = df.merge(th[["k", "total"]].rename(columns={"total": "TH_Transito_Cupos"}), on="k", how="left")
print(f"Enlazados TH_Transito: {df['TH_Transito_Cupos'].notna().sum()} / {len(df)}")

---
## 5. Cupos UDS 31/12/2025 (`UDS_31122025`)

Se integran las columnas `CuposUDS_31122025` y `CuposServicioUDS_31122025`
desde la hoja `UDS_31122025`, desduplicada por `CodigoUnidadServicioUDS`.

In [ ]:
uds25 = pd.read_excel(UDS_FILE, sheet_name="UDS_31122025")
uds25["k"] = uds25["CodigoUnidadServicioUDS"].astype(str).str.strip()
uds25 = uds25.drop_duplicates(subset="k", keep="first")
print(f"UDS_31122025: {len(uds25)} registros únicos (de {len(uds25) + uds25.duplicated(subset='k').sum()} brutos)")

df = df.merge(uds25[["k", "CuposUDS_31122025", "CuposServicioUDS_31122025"]], on="k", how="left")
print(f"CuposUDS_31122025 enlazados: {df['CuposUDS_31122025'].notna().sum()} / {len(df)}")
print(f"CuposServicioUDS_31122025 enlazados: {df['CuposServicioUDS_31122025'].notna().sum()} / {len(df)}")

---
## 6. Campos derivados

Se calculan campos que no existen directamente en los datos fuente:
- **NIT_EC**: NIT del contratista limpio (solo dígitos).
- **NumeroContrato_clean**: Número de contrato como string limpio.
- **Modalidad_calc**: Modalidad 2026 inferida desde `Abrev` → si es abreviatura conocida
  (HCB → FAMILIAR Y COMUNITARIA, BV/JC → INSTITUCIONAL), o desde `Servicio` por palabras clave,
  o fallback a `Tipo_Modalidad`.

In [ ]:
def clean_nit(x):
    if pd.isna(x): return None
    s = str(x).replace(',', '.').split('.')[0]
    s = ''.join(c for c in s if c.isdigit())
    return s if s else None

df["NIT_EC"] = df["NumeroDocumentoEC"].apply(clean_nit)
df["NumeroContrato_clean"] = df["NumeroContrato"].astype(str).str.strip()

Abrev_map = {
    "HCB": "FAMILIAR Y COMUNITARIA",
    "FAMI": "FAMILIAR Y COMUNITARIA",
    "HCB_SA": "FAMILIAR Y COMUNITARIA",
    "BV": "INSTITUCIONAL",
    "JC": "INSTITUCIONAL",
}

def modalidad_func(row):
    if pd.notna(row["Abrev"]) and row["Abrev"] in Abrev_map:
        return Abrev_map[row["Abrev"]]
    s = str(row["Servicio"]).upper() if pd.notna(row["Servicio"]) else ""
    if "PROPIA" in s or "INTERCULTURAL" in s:
        return "PROPIA E INTERCULTURAL"
    if "INSTITUCIONAL" in s or "CDI" in s or "CENTRO DE DESARROLLO" in s:
        return "INSTITUCIONAL"
    if "MODELO PROPIO" in s or "MAI" in s:
        return "MODELO PROPIO"
    if "FAMILIAR" in s or "COMUNITARIA" in s or "HCB" in s:
        return "FAMILIAR Y COMUNITARIA"
    return row["Abrev"] if pd.notna(row["Abrev"]) else None

df["Modalidad_calc"] = df.apply(modalidad_func, axis=1).fillna(df["Tipo_Modalidad"])
print(f"Modalidad_calc completa: {df['Modalidad_calc'].notna().sum()} / {len(df)}")
print(f"Modalidades asignadas: {df['Modalidad_calc'].value_counts().to_dict()}")

---
## 7. Mapeo a estructura ZonificacionPais

Se construye un DataFrame vacío con las 97 columnas no monetarias del original.
Luego se mapean las columnas fuente → destino usando un diccionario con normalización
acentos-insensible. Las columnas adicionales que no se mapean explícitamente se dejan vacías.

In [ ]:
ORIG_FILE = os.path.join(DIR_BASE, "Metaverso 2026.xlsx")
meta = pd.read_excel(ORIG_FILE, sheet_name="ZonificacionPais")
MONETARY = {
    " APALANCAMIENTO NUEVO DOS DIAS ",
    " VALOR TOTAL 2026 NUEVO ",
    " VALOR SOLO 2026 ",
    "APALANCA MAS VIGENCIA NUEVO 2026",
    "Costo total hasta Julio ajuste operador alimentos",
}
meta_cols = [c for c in list(meta.columns) if c not in MONETARY]
print(f"Columnas objetivo (no monetarias): {len(meta_cols)}")

In [ ]:
def normalize(s):
    return unicodedata.normalize('NFKD', str(s)).encode('ASCII', 'ignore').decode('ASCII').lower().replace(' ', '')

M = {
    "Regional UDS": "Regional_UDS",
    "Centro Zonal UDS": "Centro_Zonal_UDS",
    "Municipio UDS": "Municipio_UDS",
    "ZONA 2026": "ZONA",
    "Codigo Unidad Servicio UDS": "Codigo_UDS",
    "Unidad Servicio UDS": "UDS",
    "SERVICIO\n2026": "Servicio",
    "Cupos a Programar 2026": "Cupos",
    "Cantidad Madres\nComunitarias en la UDS": "Madres_Unds",
    "COMPONENTE": "Componente_para_la_UDS",
    "TIPO DE\nCONTRATACI\u00d3N 2026": "LITERAL_DE_CONTRATACION",
    "Departamento UDS": "DepartamentoUDS",
    "Barrio UDS": "BarrioUDS",
    "Direccion UDS": "DireccionUDS",
    "Estado UDS": "EstadoUDS",
    "Cupos UDS ofertados 2025": "CuposUDS",
    "CONTRATISTA 2026": "EntidadContratista",
    "NIT CONTRATISTA 2026": "NIT_EC",
    "Propiedad de\nla infraestructura ": "NombrePropiedadInfraestructura",
    "Departamento Entidad Contratista\n2025": "DepartamentoEC",
    "Municipio Entidad Contratista\n2025": "MunicipioEC",
    "Codigo Municipio Entidad Contratista\n2025": "CodigoMunicipioEC",
    "Zona Ubicaci\u00f3n UDS": "ZonaUbicacionUDS",
    "Vigencia UDS": "VigenciaServicio",
    "CodigoRegional": "CodigoRegionalUDS",
    "Modalidad 2026": "Modalidad_calc",
    "NIT_EntidadContratista": "EntidadContratista",
    "Modalidad\n2025": "Modalidad_calc",
    "Servicio 2025": "Servicio",
}

out = pd.DataFrame(index=range(len(df)), columns=meta_cols)
mn = {normalize(c): c for c in meta_cols}
asg = set()

for mc, sc in M.items():
    n = normalize(mc)
    if n in mn:
        out[mn[n]] = df[sc].values
        asg.add(mn[n])

# Fallback manual para Número Contrato
for c in meta_cols:
    ca = "".join(ch for ch in c if ch.isascii()).lower().strip()
    if "contrato" in ca and ca.startswith("n") and c not in asg:
        out[c] = df["NumeroContrato_clean"].values
        asg.add(c)

print(f"Columnas mapeadas: {len(asg)} / {len(meta_cols)}")
print(f"Columnas vacías (no mapeadas): {len(meta_cols) - len(asg)}")

---
## 8. Columnas nuevas

Se agregan las 3 columnas que diferencian este archivo del original:
- `CuposUDS_31122025`: cupos aprobados UDS al 31/12/2025
- `CuposServicioUDS_31122025`: cupos por servicio UDS al 31/12/2025
- `TH_Transito_Cupos`: cupos de tránsito (TH)

In [ ]:
out["CuposUDS_31122025"] = df["CuposUDS_31122025"].values
out["CuposServicioUDS_31122025"] = df["CuposServicioUDS_31122025"].values
out["TH_Transito_Cupos"] = df["TH_Transito_Cupos"].values

# Limpieza de NIT en columnas que lo contengan (excepto NIT_EntidadContratista que guarda nombre)
for c in out.columns:
    if "NIT_EntidadContratista" in str(c):
        continue
    ca = "".join(ch for ch in c if ch.isascii()).lower().strip()
    if "nit" in ca and "contratista" in ca:
        out[c] = out[c].apply(
            lambda x: str(int(x)) if pd.notna(x) and str(x).replace(".0", "").isdigit()
            else (str(x) if pd.notna(x) else None)
        )

print(f"DataFrame final: {len(out)} filas, {len(out.columns)} columnas")

---
## 9. Fórmulas

Se agregan fórmulas en la columna `Unnamed: 101` (columna de validación del original).
La fórmula `=+EXACT(CR{row},CF{row})` compara las columnas `RIESGO TRANSPARENCIA` y
`Servicio ajustado + nuevas`. En el original la fórmula era `=+EXACT(CW{row},CJ{row})`
pero las letras de columna cambiaron al eliminar las 5 columnas monetarias.

In [ ]:
# Las fórmulas se escriben directamente en el archivo Excel vía openpyxl
# (no se pueden representar como valores en un DataFrame de pandas)
print("Las fórmulas se agregarán en el paso de guardado con openpyxl.")
print(f"RIESGO TRANSPARENCIA -> col CR (0-indexed: 95)")
print(f"Servicio ajustado + nuevas -> col CF (0-indexed: 83)")

---
## 10. Guardado con openpyxl (fórmulas + columnas ocultas)

Se guarda el DataFrame con pandas, luego se reabre con openpyxl para:
1. Escribir las fórmulas en `Unnamed: 101`
2. Ocultar las mismas columnas que están ocultas en el archivo original

> **Nota**: Este paso puede tomar ~2 minutos por las 55k fórmulas.

In [ ]:
OUTPUT_FILE = os.path.join(DIR_BASE, "ZONIFICACIONPAIS_RECONSTRUIDA_2026.xlsx")

out.to_excel(OUTPUT_FILE, index=False)
print(f"Archivo guardado: {OUTPUT_FILE}")
print(f"Filas: {len(out)}, Columnas: {len(out.columns)}")

In [ ]:
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
import time

wb = load_workbook(OUTPUT_FILE)
ws = wb.active

# --- Fórmulas ---
formula_col = None
for col in range(1, ws.max_column + 1):
    v = ws.cell(1, col).value
    if v and 'Unnamed' in str(v):
        formula_col = col
        break

riesgo_letter = get_column_letter(96)   # RIESGO TRANSPARENCIA (0-indexed 95)
servicio_letter = get_column_letter(84)  # Servicio ajustado + nuevas (0-indexed 83)

t0 = time.time()
batch_size = 5000
for bs in range(2, ws.max_row + 1, batch_size):
    be = min(bs + batch_size - 1, ws.max_row)
    for r in range(bs, be + 1):
        ws.cell(r, formula_col).value = f"=+EXACT({riesgo_letter}{r},{servicio_letter}{r})"
    wb.save(OUTPUT_FILE)
print(f"Fórmulas escritas: {ws.max_row - 1} en {time.time()-t0:.1f}s")

In [ ]:
# --- Columnas ocultas ---
# Columnas ocultas en el original (1-based):
# 27-37, 45-64, 66, 68-75, 78-84, 86, 92-95, 99, 101-102
# Ajuste: se removieron las monetarias 38-41 (shift -4 para cols >= 42)
#          y la monetaria 90 (shift -5 para cols >= 91)

hidden_orig = (
    list(range(27, 38)) +
    list(range(45, 65)) +
    [66] +
    list(range(68, 76)) +
    list(range(78, 85)) +
    [86] +
    list(range(92, 96)) +
    [99, 101, 102]
)

hidden_output = set()
for h in hidden_orig:
    if h >= 91:
        shift = 5
    elif h >= 42:
        shift = 4
    else:
        shift = 0
    out_col = h - shift
    if 1 <= out_col <= ws.max_column:
        hidden_output.add(out_col)

# No ocultar las 3 nuevas columnas (98-100)
for nc in [98, 99, 100]:
    hidden_output.discard(nc)

for col_num in sorted(hidden_output):
    ws.column_dimensions[get_column_letter(col_num)].hidden = True

wb.save(OUTPUT_FILE)
wb.close()
print(f"Columnas ocultas: {len(hidden_output)}")
print(f"Archivo final: {OUTPUT_FILE}")

---
# Enfoque alternativo: ACTUALIZACIÓN del ZonificacionPais existente

En lugar de reconstruir desde cero, este enfoque **actualiza** la hoja `ZonificacionPais`
del archivo `Metaverso 2026.xlsx` original:
- Las filas existentes con código UDS que coinciden con los nuevos datos → se actualizan (`Actualizado`)
- Las filas existentes sin coincidencia → se conservan intactas (`No actualizable`)
- Los nuevos UDS de los insumos que no están en el original → se agregan al final (`Nuevo`)
- Se agrega una columna `Estado_Actualizacion` para identificar cada caso
- Las 5 columnas monetarias del original se conservan (no se reconstruyen, no hay fuente)

> Ejecute las celdas a continuación en lugar de las secciones 6–10 si prefiere este enfoque.

In [ ]:
print("="*60)
print("ACTUALIZACIÓN del ZonificacionPais existente")
print("="*60)

# Cargar el ZonificacionPais original
OUTPUT_UPDATE = os.path.join(DIR_BASE, "ZONIFICACIONPAIS_ACTUALIZADA_2026.xlsx")
orig = pd.read_excel(ORIG_FILE, sheet_name="ZonificacionPais")
ORIG_COLS = list(orig.columns)
print(f"Original: {len(orig)} filas, {len(ORIG_COLS)} columnas")

def normalize(s):
    return unicodedata.normalize("NFKD", str(s)).encode("ASCII", "ignore").decode("ASCII").lower().replace(" ", "").replace("\n", "")

MONETARY = {
    " APALANCAMIENTO NUEVO DOS DIAS ",
    " VALOR TOTAL 2026 NUEVO ",
    " VALOR SOLO 2026 ",
    "APALANCA MAS VIGENCIA NUEVO 2026",
    "Costo total hasta Julio ajuste operador alimentos",
}

orig_norm = {normalize(c): c for c in ORIG_COLS if c not in MONETARY}
orig["k"] = orig["Codigo Unidad Servicio UDS"].astype(str).str.strip()()
orig["k"] = orig["k"].replace(["nan", "None", ""], None)
orig_codes = set(orig.loc[orig["k"].notna(), "k"].unique())

In [ ]:
# Mapeo de columnas fuente -> destino (mismo que rebuild)
UPDATE_MAP = [
    ("Regional_UDS",               "Regional UDS"),
    ("Centro_Zonal_UDS",           "Centro Zonal UDS"),
    ("Municipio_UDS",              "Municipio UDS"),
    ("ZONA",                       "ZONA 2026"),
    ("Codigo_UDS",                 "Codigo Unidad Servicio UDS"),
    ("UDS",                        "Unidad Servicio UDS"),
    ("LITERAL_DE_CONTRATACION",    "TIPO DE CONTRATACION 2026"),
    ("Servicio",                   "SERVICIO 2026"),
    ("Cupos",                      "Cupos a Programar 2026"),
    ("Madres_Unds",                "Cantidad Madres Comunitarias en la UDS"),
    ("Componente_para_la_UDS",     "COMPONENTE"),
    ("DepartamentoUDS",            "Departamento UDS"),
    ("BarrioUDS",                  "Barrio UDS"),
    ("DireccionUDS",               "Direccion UDS"),
    ("EstadoUDS",                  "Estado UDS"),
    ("CuposUDS",                   "Cupos UDS ofertados 2025"),
    ("EntidadContratista",         "CONTRATISTA 2026"),
    ("NIT_EC",                     "NIT CONTRATISTA 2026"),
    ("NumeroContrato_clean",       "Numero Contrato"),
    ("TipoOrganizacionEC",         "Tipo de Organizacion"),
    ("DepartamentoEC",             "Departamento Entidad Contratista 2025"),
    ("MunicipioEC",                "Municipio Entidad Contratista 2025"),
    ("CodigoMunicipioEC",          "Codigo Municipio Entidad Contratista 2025"),
    ("ZonaUbicacionUDS",           "Zona Ubicacion UDS"),
    ("VigenciaServicio",           "Vigencia UDS"),
    ("CodigoRegionalUDS",          "CodigoRegional"),
    ("NombrePropiedadInfraestructura", "Propiedad de la infraestructura"),
    ("Modalidad_calc",             "Modalidad 2026"),
    ("EntidadContratista",         "NIT_EntidadContratista"),  # store name despite column name
    ("Servicio",                   "Servicio 2025"),
    ("Modalidad_calc",             "Modalidad 2025"),
]

resolved = []
for src, dst in UPDATE_MAP:
    n = normalize(dst)
    if n in orig_norm:
        resolved.append((src, orig_norm[n]))
    else:
        print(f"  AVISO: '{dst}' no encontrado en ZonificacionPais")
print(f"Mapeos resueltos: {len(resolved)}")

In [ ]:
# Merge vectorizado: nuevas fuentes vs original
NEW_UPDATE_COLS = list({src for src,_ in resolved}) + ["CuposUDS_31122025","CuposServicioUDS_31122025","TH_Transito_Cupos"]
NEW_UPDATE_COLS = [c for c in NEW_UPDATE_COLS if c in df.columns]

df_dedup = df.drop_duplicates(subset="k", keep="first").set_index("k")
dup = df_dedup[NEW_UPDATE_COLS].rename(columns={c:f"_n_{c}" for c in NEW_UPDATE_COLS})
merged = orig[["k"]].merge(dup, left_on="k", right_index=True, how="left")

# Clasificar estado de actualización
new_keys_set = set(df["k"].unique())
audit = pd.Series("No actualizable", index=orig.index, dtype="string")
audit[merged["k"].isin(new_keys_set)] = "Actualizado"
audit[orig["k"].isna()] = "Sin codigo"
orig["Estado_Actualizacion"] = audit

# Actualizar columnas mapeadas (solo donde hay valor nuevo)
for src, dst in resolved:
    nc = f"_n_{src}"
    if nc in merged.columns:
        mask = merged[nc].notna()
        orig.loc[mask.values, dst] = merged.loc[mask.values, nc].values

# Agregar 3 columnas nuevas
NEW_COLS = ["CuposUDS_31122025","CuposServicioUDS_31122025","TH_Transito_Cupos"]
for c in NEW_COLS:
    nc = f"_n_{c}"
    orig[c] = orig.get(c) if nc not in merged.columns else merged[nc]

print(f"Actualizados: {(audit=='Actualizado').sum():>6}")
print(f"No actualizables: {(audit=='No actualizable').sum():>6}")
print(f"Sin codigo: {(audit=='Sin codigo').sum():>6}")

In [ ]:
# Agregar nuevos UDS (existen en fuentes pero no en original)
exist = set(orig.loc[orig["k"].notna(), "k"].unique())
nuevos = new_keys_set - exist
print(f"Nuevos UDS a agregar: {len(nuevos)}")

if nuevos:
    nr = df_dedup.loc[df_dedup.index.isin(nuevos)].copy()
    ndf = pd.DataFrame(index=range(len(nr)), columns=ORIG_COLS, dtype=object)
    for src, dst in resolved:
        if src in nr.columns:
            ndf[dst] = nr[src].values
    for c in NEW_COLS:
        if c in nr.columns:
            ndf[c] = nr[c].values
    ndf["Estado_Actualizacion"] = "Nuevo"
    orig = pd.concat([orig, ndf], ignore_index=True)
    print(f"  {len(nuevos)} nuevos UDS agregados")

# Limpiar NITs (igual que rebuild)
for c in orig.columns:
    if "NIT_EntidadContratista" in str(c): continue
    ca = "".join(ch for ch in c if ch.isascii()).lower().strip()
    if "nit" in ca and "contratista" in ca:
        orig[c] = orig[c].apply(lambda x: str(int(x)) if pd.notna(x) and str(x).replace(".0","").isdigit() else (str(x) if pd.notna(x) else None))

In [ ]:
# Armar DataFrame final con orden de columnas
final_cols = [c for c in ORIG_COLS if c not in MONETARY and c != "k"]
for c in NEW_COLS:
    if c not in orig.columns: orig[c] = None

unnamed = None
for c in final_cols:
    if str(c).startswith("Unnamed"):
        unnamed = c; final_cols.remove(c); break

final_cols += [c for c in NEW_COLS if c not in final_cols] + ["Estado_Actualizacion"]
if unnamed: final_cols.append(unnamed)
final_cols = [c for c in final_cols if c in orig.columns]
out = orig[final_cols]

print(f"DataFrame final: {len(out)} filas, {len(final_cols)} columnas")

In [ ]:
# Guardado con openpyxl + fórmulas en Unnamed: 101
out.to_excel(OUTPUT_UPDATE, index=False)
print(f"Archivo guardado: {OUTPUT_UPDATE}")

from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
import time

wb = load_workbook(OUTPUT_UPDATE)
ws = wb.active

# Encontrar columnas Riesgo y Unnamed en el output final
riesgo_idx = None
unnamed_idx = None
for i, c in enumerate(final_cols):
    if normalize(c) == normalize("RIESGO TRANSPARENCIA"):
        riesgo_idx = i
    elif str(c).startswith("Unnamed"):
        unnamed_idx = i

if riesgo_idx is not None and unnamed_idx is not None:
    riesgo_letter = get_column_letter(riesgo_idx + 1)
    target_letter = get_column_letter(unnamed_idx + 1)
    print(f"Riesgo: col {riesgo_letter}, Unnamed: col {target_letter}")
    
    t0 = time.time()
    batch_size = 5000
    for bs in range(2, ws.max_row + 1, batch_size):
        be = min(bs + batch_size - 1, ws.max_row)
        for r in range(bs, be + 1):
            ws.cell(r, unnamed_idx + 1).value = f"=+EXACT({riesgo_letter}{r},CF{r})"
        wb.save(OUTPUT_UPDATE)
    print(f"Fórmulas escritas: {ws.max_row - 1} en {time.time()-t0:.1f}s")
else:
    print("AVISO: No se encontraron columnas Riesgo/Unnamed")

wb.save(OUTPUT_UPDATE)
wb.close()
print(f"Archivo final: {OUTPUT_UPDATE}")

---
## 11. Actualización complementaria: Integrales Zonificación

Los registros marcados como **No actualizable** no tenían correspondencia en CONCAT.
Sin embargo, algunos de esos códigos UDS existen en la hoja `ZONIFICACIÓN- PEDAGOGICO`
del archivo `consolidacion_matriz_integrales_28042026_COPIA_SEGURA.xlsx`.

Este paso complementario busca esos códigos y actualiza sus columnas
con los datos de Integrales, cambiando su estado a **Actualizado x integrales**.

In [ ]:
import subprocess
import sys

script = os.path.join(DIR_BASE, "..", "..", "src", "metaverso_pipeline", "update_zonificacionpais_integrales.py")
script = os.path.normpath(script)
if not os.path.exists(script):
    script = os.path.join(os.getcwd(), "src", "metaverso_pipeline", "update_zonificacionpais_integrales.py")

print(f"Ejecutando: {script}")
result = subprocess.run([sys.executable, script], capture_output=False)
print(f"\nCodigo de retorno: {result.returncode}")

---
## 12. Análisis de anomalías — valores vacíos

A continuación se documentan todas las celdas con valores vacíos en columnas que
**teóricamente deberían estar llenas**. Se clasifican en:

| Tipo | Descripción |
|------|-------------|
| **A** | Códigos UDS sin correspondencia en el maestro UDS_15052026 |
| **B** | Códigos TH_Transito sin correspondencia en CONCAT |
| **C** | Maestro UDS con datos faltantes (ej. EntidadContratista) |
| **D** | Columnas del original que se vaciaron porque su fuente no existe en los insumos |
| **E** | Columnas nuevas con datos faltantes |

### Tipo A: Códigos UDS sin maestro

In [ ]:
no_uds = df_cat[~df_cat["k"].isin(uds["k"].unique())]
print(f"Códigos UDS en CONCAT sin correspondencia en UDS_15052026: {len(no_uds)}")
print(f"  - Integrales: {len(no_uds[no_uds['Tipo_Modalidad']=='INTEGRAL'])}")
print(f"  - Comunitarios: {len(no_uds[no_uds['Tipo_Modalidad']=='COMUNITARIO'])}")
print(f"\n  Ejemplos: {no_uds['Codigo_UDS'].head(10).tolist()}")

# Impacto en columnas
impact_cols = [
    "Departamento UDS", "Municipio UDS", "Centro Zonal UDS",
    "Direccion UDS", "Estado UDS", "Vigencia UDS",
    "Cupos UDS ofertados 2025", "CONTRATISTA 2026", "NIT CONTRATISTA 2026",
]
for c in impact_cols:
    if c in out.columns:
        empty = out[c].isna().sum()
        if empty > 0:
            print(f"  {c:<45} {empty:>6} vacíos  ({empty/len(out)*100:.1f}%)")

### Tipo B: Códigos TH_Transito sin correspondencia en CONCAT

In [ ]:
th_not_in_cat = th[~th["k"].isin(df_cat["k"].unique())]
print(f"Códigos TH_Transito NO encontrados en CONCAT: {len(th_not_in_cat)}")
print(f"Cupos total no asignados: {th_not_in_cat['total'].sum():.0f}")
print(f"\n  Ejemplos: {th_not_in_cat['codigounidadservicio'].head(10).tolist()}")

### Tipo C: Maestro UDS con datos faltantes

In [ ]:
print("UDS_15052026: registros con campos críticos vacíos")
critical_cols = ["EntidadContratista", "NumeroDocumentoEC", "DepartamentoUDS", "MunicipioUDS"]
for c in critical_cols:
    if c in uds.columns:
        empty = uds[c].isna().sum()
        pct = empty / len(uds) * 100
        print(f"  {c:<30} {empty:>6} vacíos  ({pct:.2f}%)")

# Impacto: 21 registros sin EntidadContratista se propagan a 1,343 filas en el output
no_ec = uds[uds["EntidadContratista"].isna()]
print(f"\n21 UDS sin contratista -> impactan {len(df[df['k'].isin(no_ec['k'])])} filas en el output")
print(f"  Códigos afectados: {no_ec['CodigoUnidadServicioUDS'].head(21).tolist()}")

### Tipo D: Columnas del original sin fuente

Estas columnas existían en el ZonificacionPais original pero no tienen
correspondencia en los datos fuente (CONCAT + UDS). Todas quedaron 100% vacías.

In [ ]:
fully_empty = [c for c in out.columns if out[c].isna().sum() == len(out) and c not in MONETARY]
partially_empty = [c for c in out.columns if out[c].isna().sum() > 0 and out[c].isna().sum() < len(out)]

print(f"Columnas 100% vacías (sin fuente): {len(fully_empty)}")
for c in fully_empty[:20]:
    print(f"  - {repr(c[:60])}")
if len(fully_empty) > 20:
    print(f"  ... y {len(fully_empty) - 20} más")

print(f"\nColumnas parcialmente vacías: {len(partially_empty)}")
for c in partially_empty:
    empty = out[c].isna().sum()
    pct = empty / len(out) * 100
    print(f"  - {repr(c[:55]):<57} {empty:>6} vacíos ({pct:5.1f}%)")

### Tipo E: Columnas nuevas con datos faltantes

In [ ]:
new_cols = ["CuposUDS_31122025", "CuposServicioUDS_31122025", "TH_Transito_Cupos"]
for c in new_cols:
    empty = out[c].isna().sum()
    pct = empty / len(out) * 100
    print(f"{c:<35} {empty:>6} vacíos  ({pct:.1f}%)")
    if empty > 0:
        # Mostrar % por tipo de modalidad
        tmp = df[["Tipo_Modalidad", c]].copy()
        print(f"  Por modalidad:")
        for mod in ["COMUNITARIO", "INTEGRAL"]:
            sub = tmp[tmp["Tipo_Modalidad"] == mod]
            e = sub[c].isna().sum()
            print(f"    {mod:<15} {e:>6}/{len(sub):>6} vacíos ({e/len(sub)*100:.1f}%)")

---
## 13. Resumen de anomalías

| # | Anomalía | Cantidad | Impacto | Causa raíz |
|---|----------|----------|---------|------------|
| 1 | Códigos CONCAT sin UDS | **1,326** (2.4%) | Todas las columnas del maestro vacías (Dpto, Mpio, Dirección, Estado, Contratista, NIT, etc.) | Códigos UDS en zonificación que no existen en el maestro UDS_15052026. 1,309 son Integrales, 17 Comunitarios. |
| 2 | TH_Transito sin CONCAT | **692** (10%) | 1,103 cupos de tránsito no asignados | Códigos en el registro de tránsito que no aparecen en la zonificación. Posiblemente UDS dadas de baja o fuera del alcance. |
| 3 | Maestro sin contratista | **21** UDS → **1,343** filas | `NIT_EntidadContratista`, `CONTRATISTA 2026`, `NIT CONTRATISTA 2026` vacíos | 21 registros UDS no tienen `EntidadContratista` ni `NumeroDocumentoEC`. Se propagan a 1,343 filas por códigos repetidos. |
| 4 | Columnas sin fuente | **~60** columnas | 100% vacías | Columnas de análisis y decisión (Sanciones, Modalidad por concepto, ZonaControl, etc.) que eran llenadas manualmente en el original y no tienen fuente en los datasets actuales. **Comportamiento esperado.** |
| 5 | TH_Transito_Cupos vacíos | **49,358** (88.8%) | Sin cupos de tránsito para la mayoría de UDS | Solo 6,233 UDS tienen registro de tránsito. El 88.8% de filas queda vacío naturalmente. **Comportamiento esperado.** |
| 6 | `CuposUDS_31122025` vacíos | **220** (0.4%) | Sin cupos al 31/12/2025 | Códigos UDS en CONCAT que no están en UDS_31122025. Subconjunto de los 1,326 del #1. |
| 7 | `CuposServicioUDS_31122025` vacíos | **743** (1.3%) | Sin cupos por servicio al 31/12/2025 | Similar al #6; algunos códigos tienen CuposUDS pero no CuposServicioUDS. |
| 8 | `Cupos UDS ofertados 2025` vacíos | **1,326** (2.4%) | Mismos 1,326 del #1 | Es la columna del ZonificacionPais que se llena con `CuposUDS` del maestro. Misma causa raíz. |

---
## 14. Comparación con el original

A continuación se comparan las columnas que **mejoraron** vs **empeoraron**
respecto al archivo Metaverso 2026 original.

In [ ]:
orig = pd.read_excel(ORIG_FILE, sheet_name="ZonificacionPais")

improved = []
worsened = []
for c in out.columns:
    if c in orig.columns and c not in MONETARY:
        orig_e = orig[c].isna().sum()
        new_e = out[c].isna().sum()
        delta = new_e - orig_e
        if delta < 0:
            improved.append((c, orig_e, new_e, -delta))
        elif delta > 0:
            worsened.append((c, orig_e, new_e, delta))

print("=== Columnas que MEJORARON (menos vacíos que el original) ===")
print(f"{'Columna':<50} {'Original':>8} {'Nuevo':>8} {'Mejora':>8}")
print("-" * 76)
for c, o, n, d in sorted(improved, key=lambda x: -x[3])[:15]:
    print(f"{repr(c[:48]):<50} {o:>8} {n:>8} {d:>8}")

print(f"\n=== Columnas que EMPEORARON (más vacíos que el original) ===")
print(f"{'Columna':<50} {'Original':>8} {'Nuevo':>8} {'Diferencia':>10}")
print("-" * 78)
for c, o, n, d in sorted(worsened, key=lambda x: -x[3])[:20]:
    print(f"{repr(c[:48]):<50} {o:>8} {n:>8} {d:>10}")

---
## 15. Conclusiones

### Hallazgos principales

1. **1,326 códigos UDS** de la zonificación 2026 no existen en el maestro UDS_15052026
   (2.4% del total). Esto afecta 23 columnas del maestro que quedan sin datos.
   - 1,309 son de modalidad INTEGRAL (posiblemente códigos nuevos creados después del corte)
   - 17 son COMUNITARIO

2. **692 códigos** del registro de tránsito TH no aparecen en la zonificación 2026,
   dejando 1,103 cupos sin asignar.

3. **60 columnas** quedaron 100% vacías porque no existe fuente de datos para ellas
   en los insumos actuales. Son columnas de decisión y análisis que se llenaban manualmente
   en el proceso de zonificación original.

### Recomendaciones

- Validar los 1,326 códigos no enlazados con el equipo de planta física para determinar
  si son UDS válidas que deben agregarse al maestro o errores en la zonificación.
- Revisar los 692 códigos TH sin correspondencia para determinar si representan
  UDS activas que deberían estar en la zonificación.
- Para las 60 columnas vacías, evaluar si es necesario buscar fuentes alternativas
  (histórico de Metaverso 2025, archivos de sanciones, etc.) o si se aceptan como vacías
  en el nuevo proceso.